# Two-Fluid LCDM TT Spectrum: Colab Starter Notebook

This notebook downloads the simplified two-fluid LCDM code, compiles the C++ solver, plots the CMB temperature power spectrum, and evaluates the negative log likelihood for the included mock data.

The model parameter vector is

```text
theta = [log10(10^9 A_s), omega_cdm, omega_b, H0, n_s]
```

where `H0` is in km/s/Mpc, while `omega_cdm` and `omega_b` are physical densities: `Omega_cdm h^2` and `Omega_b h^2`.

In [ ]:
# Download or update the repository.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/tsmith2/GGI_cosmology_notebooks.git"
REPO_DIR = Path("/content/GGI_cosmology_notebooks")
PACKAGE_DIR = REPO_DIR / "two_fluid_LCDM_colab"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(PACKAGE_DIR)
print("Working in", Path.cwd())

In [ ]:
# Compile the C++ two-fluid TT solver.
subprocess.run(["make", "-C", "cpp"], check=True)

In [ ]:
# Import the teaching wrapper.
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(PACKAGE_DIR / "python"))

from two_fluid_LCDM_colab import (
    load_mock_data,
    plot_power_spectrum,
    neg_log_likelihood_lcdm,
    model_bandpowers,
    tt_spectrum,
)

In [ ]:
# Choose a fiducial LCDM parameter vector explicitly.
# theta = [log10(10^9 A_s), omega_cdm, omega_b, H0, n_s]
theta = np.array([np.log10(2.1), 0.1201, 0.0223, 67.0, 0.965])

data = load_mock_data()
print("theta =", theta)
print("number of mock bandpowers =", len(data["d_obs"]))

In [ ]:
# Plot the model TT spectrum and the included mock bandpowers.
fig, ax = plot_power_spectrum(theta, data=data)
plt.show()

In [ ]:
# Compute the negative log likelihood for this model.
nll = neg_log_likelihood_lcdm(theta, data)
print(f"-log likelihood = {nll:.3f}")

## Try a different value of H0

The next cell changes only `H0`, recomputes the model, and compares its likelihood to the fiducial model. This is a quick way to see how the mock data respond to changing one cosmological parameter.

In [ ]:
theta_alt = theta.copy()
theta_alt[3] = 72.0   # H0 in km/s/Mpc

nll_fid = neg_log_likelihood_lcdm(theta, data)
nll_alt = neg_log_likelihood_lcdm(theta_alt, data)

print(f"fiducial H0 = {theta[3]:.1f}, -log L = {nll_fid:.3f}")
print(f"alternate H0 = {theta_alt[3]:.1f}, -log L = {nll_alt:.3f}")

fig, ax = plot_power_spectrum(theta, data=data)
spec_alt = tt_spectrum(theta_alt)
ax.plot(spec_alt.ell, spec_alt.dell_over_2pi, color="royalblue", lw=1.2, label="H0 = 72")
ax.legend(frameon=False)
plt.show()

## A simple one-parameter scan

This is not a full parameter fit. It simply scans over `H0` while keeping the other parameters fixed.

In [ ]:
H0_values = np.linspace(60.0, 75.0, 16)
nll_values = []

for H0 in H0_values:
    trial = theta.copy()
    trial[3] = H0
    nll_values.append(neg_log_likelihood_lcdm(trial, data))

nll_values = np.array(nll_values)

plt.figure(figsize=(6.0, 4.0), dpi=140)
plt.plot(H0_values, nll_values - np.min(nll_values), "o-", color="black")
plt.xlabel(r"$H_0$ [km/s/Mpc]")
plt.ylabel(r"$-\log L - \min(-\log L)$")
plt.title("One-parameter scan with other parameters fixed")
plt.grid(alpha=0.25)
plt.show()

best = H0_values[np.argmin(nll_values)]
print(f"Best H0 on this coarse grid: {best:.2f} km/s/Mpc")

## Notes

- The first model evaluation can take longer because the code creates a reusable Bessel-function cache in `two_fluid_LCDM_colab/cache/`.
- Later evaluations reuse this cache and should be faster.
- The included mock data are stored in `two_fluid_LCDM_colab/data/mock_tt_planck_like.npz`.
- This notebook intentionally does not run an MCMC sampler; it is meant as a lightweight starting point for plotting spectra and evaluating likelihoods.